In [ ]:

import requests
import pandas as pd
import json
import re
import time
import logging

from bs4 import BeautifulSoup
from urllib.parse import urljoin
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from datetime import datetime
from tqdm.auto import tqdm

# ==========================================================
# 1. CONFIGURACIÓN GENERAL
# ==========================================================

BASE_URL = "https://siempresalud.co"
URL_INDICE_COLECCIONES = f"{BASE_URL}/products"

URLS = {
    "Littmann": "https://siempresalud.co/collections/littmann",
    "Welch Allyn": "https://siempresalud.co/collections/welch-allyn",
    "Medias de compresión": "https://siempresalud.co/collections/medias-de-compresion",
    "Movilidad": "https://siempresalud.co/collections/movilidad",
    "GMD": "https://siempresalud.co/collections/gmd",
    "Diagnóstico": "https://siempresalud.co/collections/diagnostico",
    "Super Combos": "https://siempresalud.co/pages/super-combos"
}

ARCHIVO_EXCEL = "siempresalud_catalogo_completo.xlsx"

PAUSA = 1
MAX_PAGINAS = 40

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/json,*/*"
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# ==========================================================
# 2. SESIÓN HTTP CON REINTENTOS
# ==========================================================

session = requests.Session()
session.headers.update(HEADERS)

retry = Retry(
    total=4,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(max_retries=retry)

session.mount("https://", adapter)
session.mount("http://", adapter)


def solicitar(url):

    try:
        r = session.get(url, timeout=30)
        r.raise_for_status()
        return r

    except requests.RequestException as e:
        logging.warning(f"Error: {url} | {e}")
        return None


# ==========================================================
# 3. FUNCIONES AUXILIARES
# ==========================================================

def limpiar_html(texto):

    if not texto:
        return ""

    soup = BeautifulSoup(str(texto), "html.parser")
    return soup.get_text(" ", strip=True)


def precio(valor):

    if valor is None or valor == "":
        return None

    try:
        # Shopify Ajax JSON entrega precios en centavos
        return round(float(valor) / 100, 2)

    except (ValueError, TypeError):
        return None


def obtener_handle(url):

    match = re.search(
        r"/products/([^/?#]+)",
        url
    )

    return match.group(1) if match else None


def obtener_imagenes(producto):

    resultado = []

    for img in producto.get("images", []):

        if isinstance(img, dict):
            img = img.get("src", "")

        if img:
            resultado.append(img)

    return resultado


# ==========================================================
# 4. EXTRAER ENLACES DE PRODUCTOS DESDE HTML
# ==========================================================

def extraer_handles_html(url):

    respuesta = solicitar(url)

    if not respuesta:
        return set()

    soup = BeautifulSoup(respuesta.text, "lxml")

    handles = set()

    for a in soup.select("a[href]"):

        href = urljoin(BASE_URL, a.get("href", ""))

        handle = obtener_handle(href)

        if handle:
            handles.add(handle)

    return handles


def descubrir_colecciones():

    url_inicial = f"{URL_INDICE_COLECCIONES}?page=1"
    respuesta_inicial = solicitar(url_inicial)

    if not respuesta_inicial:
        raise RuntimeError(
            f"No se pudo consultar el índice de colecciones: {url_inicial}"
        )

    soup_inicial = BeautifulSoup(respuesta_inicial.text, "lxml")
    paginas = {1}

    for a in soup_inicial.select("a[href]"):
        coincidencias = re.findall(
            r"[?&]page=(\d+)",
            a.get("href", "")
        )
        paginas.update(int(pagina) for pagina in coincidencias)

    pagina_maxima = min(max(paginas), MAX_PAGINAS)
    colecciones = {}
    paginas_sin_novedades = 0

    for pagina in range(1, pagina_maxima + 1):

        if pagina == 1:
            respuesta = respuesta_inicial
        else:
            respuesta = solicitar(
                f"{URL_INDICE_COLECCIONES}?page={pagina}"
            )

        if not respuesta:
            raise RuntimeError(
                f"No se pudo consultar la página {pagina} del índice de colecciones."
            )

        soup = BeautifulSoup(respuesta.text, "lxml")
        colecciones_nuevas = 0

        for a in soup.select("a[href]"):
            href = urljoin(BASE_URL, a.get("href", ""))
            coincidencia = re.search(
                r"/collections/([^/?#]+)",
                href
            )

            if not coincidencia:
                continue

            handle = coincidencia.group(1)
            nombre = (
                a.get_text(" ", strip=True)
                or (a.get("title") or "").strip()
                or handle.replace("-", " ").title()
            )
            url_coleccion = f"{BASE_URL}/collections/{handle}"

            if handle not in colecciones:
                colecciones[handle] = (nombre, url_coleccion)
                colecciones_nuevas += 1
            elif len(nombre) > len(colecciones[handle][0]):
                colecciones[handle] = (nombre, url_coleccion)

        if colecciones_nuevas:
            paginas_sin_novedades = 0
        else:
            paginas_sin_novedades += 1

        if pagina_maxima == MAX_PAGINAS and paginas_sin_novedades >= 2:
            break

        time.sleep(PAUSA)

    if not colecciones:
        raise RuntimeError(
            "El índice de colecciones no devolvió categorías; no se actualizará el Excel."
        )

    logging.info(
        f"Índice de colecciones: {len(colecciones)} colecciones en "
        f"{pagina} páginas"
    )

    return list(colecciones.values())


# ==========================================================
# 5. EXTRAER PRODUCTOS DE CADA COLECCIÓN
# ==========================================================

def extraer_coleccion(nombre, url):

    logging.info(f"Procesando colección: {nombre}")

    handles = set()

    # Método 1: endpoint JSON de Shopify

    if "/collections/" in url:

        handle_coleccion = url.rstrip("/").split("/")[-1]

        for pagina in range(1, MAX_PAGINAS + 1):

            api_url = (
                f"{BASE_URL}/collections/"
                f"{handle_coleccion}/products.json"
                f"?limit=250&page={pagina}"
            )

            r = solicitar(api_url)

            if not r:
                break

            try:
                items = r.json().get("products", [])

            except (ValueError, AttributeError):
                break

            nuevos = 0

            for p in items:

                h = p.get("handle")

                if h and h not in handles:
                    handles.add(h)
                    nuevos += 1

            if not items or nuevos == 0:
                break

            time.sleep(PAUSA)

    # Método 2: HTML de todas las páginas
    # También sirve para descubrir productos no obtenidos
    # mediante el endpoint JSON.

    paginas_vacias = 0

    for pagina in range(1, MAX_PAGINAS + 1):

        html_url = (
            f"{url}?page={pagina}&grid_list=grid-view"
        )

        encontrados = extraer_handles_html(html_url)

        nuevos = encontrados - handles

        handles.update(encontrados)

        if not nuevos:
            paginas_vacias += 1
        else:
            paginas_vacias = 0

        if paginas_vacias >= 2:
            break

        time.sleep(PAUSA)

    logging.info(
        f"{nombre}: {len(handles)} productos únicos"
    )

    return handles


# ==========================================================
# 6. EXTRAER PRODUCTOS DE SUPER COMBOS
# ==========================================================

def extraer_super_combos(url):

    logging.info("Procesando página Super Combos")

    # La página no es una colección estándar.
    # Se extraen los enlaces de productos que aparecen en ella.

    handles = extraer_handles_html(url)

    logging.info(
        f"Super Combos: {len(handles)} productos identificados"
    )

    return handles


# ==========================================================
# 7. EXTRAER CALIFICACIONES Y RESEÑAS
# ==========================================================

def extraer_resenas(soup):

    calificacion = None
    cantidad = None

    scripts = soup.select(
        'script[type="application/ld+json"]'
    )

    for script in scripts:

        try:
            data = json.loads(script.string or script.get_text())

            elementos = data if isinstance(data, list) else [data]

            for item in elementos:

                if not isinstance(item, dict):
                    continue

                rating = item.get("aggregateRating")

                if isinstance(rating, dict):

                    calificacion = rating.get("ratingValue")

                    cantidad = rating.get(
                        "reviewCount",
                        rating.get("ratingCount")
                    )

                if isinstance(rating, list) and rating:

                    calificacion = rating[0].get("ratingValue")
                    cantidad = rating[0].get("reviewCount")

        except (ValueError, TypeError, AttributeError):
            continue

    # Conversión

    try:
        calificacion = (
            float(calificacion)
            if calificacion is not None else None
        )
    except (ValueError, TypeError):
        calificacion = None

    try:
        cantidad = (
            int(cantidad)
            if cantidad is not None else None
        )
    except (ValueError, TypeError):
        cantidad = None

    return calificacion, cantidad


# ==========================================================
# 8. EXTRAER INFORMACIÓN DE UN PRODUCTO
# ==========================================================

def extraer_producto(handle, categorias):

    url = f"{BASE_URL}/products/{handle}"

    api_url = f"{url}.js"

    r_json = solicitar(api_url)

    if not r_json:
        return None, []

    try:
        producto = r_json.json()

    except ValueError:
        return None, []

    r_html = solicitar(url)

    if not r_html:
        return None, []

    soup = BeautifulSoup(r_html.text, "lxml")

    imagenes = obtener_imagenes(producto)

    variantes = producto.get("variants", [])

    precios = [
        precio(v.get("price"))
        for v in variantes
        if v.get("price") is not None
    ]

    precios_anteriores = [
        precio(v.get("compare_at_price"))
        for v in variantes
        if v.get("compare_at_price") is not None
    ]

    disponible = any(
        v.get("available", False)
        for v in variantes
    )

    calificacion, resenas = extraer_resenas(soup)

    descripcion_html = producto.get("description", "")

    opciones = producto.get("options", [])

    nombres_opciones = [
        op.get("name", "")
        for op in opciones
    ]

    # ---------------------------
    # REGISTRO GENERAL
    # ---------------------------

    registro = {

        "ID Shopify": producto.get("id"),

        "Nombre": producto.get("title"),

        "Marca": producto.get("vendor"),

        "Tipo de producto": producto.get("type"),

        "Precio actual mínimo": min(precios) if precios else None,

        "Precio anterior mínimo": (
            min(precios_anteriores)
            if precios_anteriores else None
        ),

        "Descripción": limpiar_html(descripcion_html),

        "Imágenes": " | ".join(imagenes),

        "Cantidad de imágenes": len(imagenes),

        "Opciones de variantes": " | ".join(nombres_opciones),

        "Cantidad de variantes": len(variantes),

        "Disponibilidad general": disponible,

        "Calificación": calificacion,

        "Número de reseñas": resenas,

        "URL": url,

        "Handle": handle,

        "Categorías": " | ".join(sorted(categorias)),

        "Fecha de extracción": datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    }

    # ---------------------------
    # REGISTROS DE VARIANTES
    # ---------------------------

    filas_variantes = []

    for v in variantes:

        opciones_variante = {}

        for i, nombre in enumerate(nombres_opciones):

            opciones_variante[nombre] = v.get(
                f"option{i+1}"
            )

        fila = {

            "ID producto": producto.get("id"),

            "Producto": producto.get("title"),

            "SKU": v.get("sku"),

            "ID variante": v.get("id"),

            "Variante": v.get("title"),

            "Precio": precio(v.get("price")),

            "Precio anterior": precio(
                v.get("compare_at_price")
            ),

            "Disponibilidad": v.get("available"),

            "Inventario": v.get("inventory_quantity"),

            "Opciones": json.dumps(
                opciones_variante,
                ensure_ascii=False
            ),

            "URL": url,

            "Categorías": " | ".join(sorted(categorias))
        }

        filas_variantes.append(fila)

    return registro, filas_variantes


# ==========================================================
# 9. RECOPILAR TODAS LAS CATEGORÍAS
# ==========================================================

def recopilar_catalogo():

    mapa_productos = {}

    resumen = []

    errores = []

    urls_catalogo = dict(URLS)

    for nombre, url in descubrir_colecciones():
        if url not in urls_catalogo.values():
            clave = nombre
            if clave in urls_catalogo and urls_catalogo[clave] != url:
                handle = url.rstrip("/").split("/")[-1]
                clave = f"{nombre} ({handle})"
            urls_catalogo[clave] = url

    for nombre, url in urls_catalogo.items():

        if "/pages/" in url:

            handles = extraer_super_combos(url)

        else:

            handles = extraer_coleccion(nombre, url)

        encontrados = len(handles)

        for handle in handles:

            if handle not in mapa_productos:
                mapa_productos[handle] = set()

            mapa_productos[handle].add(nombre)

        resumen.append({
            "Categoría": nombre,
            "Productos encontrados": encontrados,
            "URL": url
        })

    logging.info(
        f"Total de productos únicos: {len(mapa_productos)}"
    )

    if not mapa_productos:
        raise RuntimeError(
            "No se encontraron productos; no se sobrescribirá el catálogo existente."
        )

    # ======================================================
    # 10. EXTRAER FICHAS INDIVIDUALES
    # ======================================================

    productos = []

    variantes = []

    for handle, categorias in tqdm(
        mapa_productos.items(),
        desc="Extrayendo fichas"
    ):

        try:

            registro, filas = extraer_producto(
                handle,
                categorias
            )

            if registro:

                productos.append(registro)

                variantes.extend(filas)

            else:

                errores.append({
                    "Handle": handle,
                    "Categorías": " | ".join(categorias),
                    "Error": "No se pudo obtener la ficha"
                })

        except Exception as e:

            errores.append({
                "Handle": handle,
                "Categorías": " | ".join(categorias),
                "Error": str(e)
            })

        time.sleep(PAUSA)

    # ======================================================
    # 11. CREAR RELACIÓN DE CATEGORÍAS
    # ======================================================

    filas_categorias = []

    for handle, categorias in mapa_productos.items():

        for categoria in categorias:

            filas_categorias.append({
                "Handle": handle,
                "Categoría": categoria,
                "URL": f"{BASE_URL}/products/{handle}"
            })

    # ======================================================
    # 12. EXPORTAR EXCEL
    # ======================================================

    with pd.ExcelWriter(
        ARCHIVO_EXCEL,
        engine="openpyxl"
    ) as writer:

        hojas = {
            "Productos": pd.DataFrame(productos),
            "Variantes": pd.DataFrame(variantes),
            "Categorías": pd.DataFrame(filas_categorias),
            "Resumen": pd.DataFrame(resumen),
            "Errores": pd.DataFrame(errores)
        }

        for nombre, df in hojas.items():

            df.to_excel(
                writer,
                sheet_name=nombre,
                index=False
            )

            ws = writer.sheets[nombre]

            ws.freeze_panes = "A2"

            if ws.max_row > 1:
                ws.auto_filter.ref = ws.dimensions

            for columna in ws.columns:

                letra = columna[0].column_letter

                max_largo = max(
                    (
                        len(str(c.value or ""))
                        for c in columna[:100]
                    ),
                    default=12
                )

                ws.column_dimensions[letra].width = min(
                    max(max_largo + 2, 12),
                    45
                )

    # ======================================================
    # 13. REPORTE FINAL
    # ======================================================

    print("\n" + "=" * 55)
    print("SCRAPING FINALIZADO")
    print("=" * 55)

    print("Productos únicos:", len(productos))
    print("Variantes:", len(variantes))
    print("Errores:", len(errores))

    print("\nProductos por categoría:")

    for item in resumen:

        print(
            item["Categoría"],
            ":", item["Productos encontrados"]
        )

    print("\nArchivo generado:", ARCHIVO_EXCEL)

    return ARCHIVO_EXCEL


# ==========================================================
# 14. EJECUTAR
# ==========================================================

archivo = recopilar_catalogo()


Extrayendo fichas:   0%|          | 0/253 [00:00<?, ?it/s]


SCRAPING FINALIZADO
Productos únicos: 253
Variantes: 614
Errores: 0

Productos por categoría:
Littmann : 38
Welch Allyn : 5
Medias de compresión : 41
Movilidad : 45
GMD : 120
Diagnóstico : 87
Super Combos : 1

Archivo generado: siempresalud_catalogo_completo.xlsx
